# 00 — Validación del dataset RDD2022

**Proyecto:** Pothole Detection Guatemala  
**Objetivo:** verificar que la copia local de RDD2022 y los manifests generados sean coherentes antes de realizar EDA, splits o entrenamiento.

Este notebook **no modifica** el dataset. Realiza controles rápidos sobre:
- dominios esperados;
- número de imágenes de `train` y `test`;
- número de XML;
- integridad de `images.csv` y `annotations.csv`;
- clases encontradas;
- consistencia entre imágenes y anotaciones.

> La validación completa de XML puede ejecutarse con `src/data/validate_dataset.py`. Ese proceso es más lento; aquí usamos principalmente los manifests ya construidos.

In [1]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

In [2]:
# Detectar la raíz del proyecto tanto si Jupyter se abrió desde la raíz
# como si se abrió desde /notebooks.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_ROOT = PROJECT_ROOT / "data" / "raw" / "RDD2022"
MANIFEST_DIR = PROJECT_ROOT / "data" / "manifests"

IMAGES_CSV = MANIFEST_DIR / "images.csv"
ANNOTATIONS_CSV = MANIFEST_DIR / "annotations.csv"

EXPECTED_DOMAINS = [
    "China_Drone",
    "China_MotorBike",
    "Czech",
    "India",
    "Japan",
    "Norway",
    "United_States",
]

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_ROOT:", DATASET_ROOT)
print("IMAGES_CSV:", IMAGES_CSV)
print("ANNOTATIONS_CSV:", ANNOTATIONS_CSV)

PROJECT_ROOT: c:\Projects\Pothole-Detection-Guatemala
DATASET_ROOT: c:\Projects\Pothole-Detection-Guatemala\data\raw\RDD2022
IMAGES_CSV: c:\Projects\Pothole-Detection-Guatemala\data\manifests\images.csv
ANNOTATIONS_CSV: c:\Projects\Pothole-Detection-Guatemala\data\manifests\annotations.csv


## 1. Comprobar que existan los archivos principales

In [3]:
checks = {
    "RDD2022": DATASET_ROOT.exists(),
    "images.csv": IMAGES_CSV.exists(),
    "annotations.csv": ANNOTATIONS_CSV.exists(),
}

pd.Series(checks, name="exists")

RDD2022            True
images.csv         True
annotations.csv    True
Name: exists, dtype: bool

In [4]:
assert DATASET_ROOT.exists(), f"No se encontró {DATASET_ROOT}"
assert IMAGES_CSV.exists(), f"No se encontró {IMAGES_CSV}"
assert ANNOTATIONS_CSV.exists(), f"No se encontró {ANNOTATIONS_CSV}"

print("Archivos principales: OK")

Archivos principales: OK


## 2. Comprobar dominios

RDD2022 completo debe contener los dominios usados en el proyecto. `China_Drone` se conserva como dominio separado porque su perspectiva de captura es distinta a la de las imágenes tomadas a nivel de carretera.

In [5]:
found_domains = sorted(
    p.name for p in DATASET_ROOT.iterdir()
    if p.is_dir() and not p.name.endswith("_INCOMPLETE")
)

domain_check = pd.DataFrame({
    "domain": EXPECTED_DOMAINS,
    "found": [d in found_domains for d in EXPECTED_DOMAINS],
})

display(domain_check)
assert domain_check["found"].all(), "Falta al menos un dominio esperado."

,domain,found
0,China_Drone,True
1,China_MotorBike,True
2,Czech,True
3,India,True
4,Japan,True
5,Norway,True
6,United_States,True


## 3. Conteo rápido de archivos por dominio

Se cuentan imágenes de `train`, XML de `train` e imágenes de `test`. La búsqueda es recursiva para soportar tanto estructuras como `Japan/Japan/train/...` como `Norway/train/...`.

In [6]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

def has_part(path, part):
    return part.lower() in [p.lower() for p in path.parts]

rows = []

for domain in EXPECTED_DOMAINS:
    root = DATASET_ROOT / domain

    train_images = [
        p for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
        and has_part(p, "train")
    ]

    test_images = [
        p for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
        and has_part(p, "test")
    ]

    train_xml = [
        p for p in root.rglob("*.xml")
        if has_part(p, "train")
    ]

    rows.append({
        "country": domain,
        "train_images": len(train_images),
        "train_xml": len(train_xml),
        "test_images": len(test_images),
    })

raw_summary = pd.DataFrame(rows)
raw_summary.loc["TOTAL"] = {
    "country": "TOTAL",
    "train_images": raw_summary["train_images"].sum(),
    "train_xml": raw_summary["train_xml"].sum(),
    "test_images": raw_summary["test_images"].sum(),
}

raw_summary

,country,train_images,train_xml,test_images
0,China_Drone,2401,2401,0
1,China_MotorBike,1977,1977,500
2,Czech,2829,2829,709
3,India,7706,7706,1959
4,Japan,10506,10506,2627
5,Norway,8161,8161,2040
6,United_States,4805,4805,1200
TOTAL,TOTAL,38385,38385,9035


In [7]:
total_train_images = int(raw_summary.iloc[:-1]["train_images"].sum())
total_train_xml = int(raw_summary.iloc[:-1]["train_xml"].sum())
total_test_images = int(raw_summary.iloc[:-1]["test_images"].sum())

print("TRAIN images:", total_train_images)
print("TRAIN XML:", total_train_xml)
print("TEST images:", total_test_images)
print("TOTAL images:", total_train_images + total_test_images)

assert total_train_images == 38385
assert total_train_xml == 38385
assert total_test_images == 9035
assert total_train_images + total_test_images == 47420

print("Conteos oficiales esperados: OK")

TRAIN images: 38385
TRAIN XML: 38385
TEST images: 9035
TOTAL images: 47420
Conteos oficiales esperados: OK


## 4. Cargar manifests

`images.csv` representa **una fila por imagen** y `annotations.csv` representa **una fila por objeto anotado**.

In [8]:
images = pd.read_csv(IMAGES_CSV)
annotations = pd.read_csv(ANNOTATIONS_CSV)

print("images.csv:", images.shape)
print("annotations.csv:", annotations.shape)

display(images.head())
display(annotations.head())

C:\Users\viank\AppData\Local\Temp\ipykernel_872\491467688.py:1: DtypeWarning: Columns (19) have mixed types. Specify dtype option on import or set low_memory=False.
  images = pd.read_csv(IMAGES_CSV)


images.csv: (38385, 20)
annotations.csv: (65712, 12)


,sample_id,country,image_path,xml_path,width,height,depth,num_objects,num_D00,num_D10,num_D20,num_D40,has_D00,has_D10,has_D20,has_D40,has_any_damage,is_empty_annotation,has_other_label,other_labels
0,China_Drone__China_Drone_000000,China_Drone,China_Drone/China_Drone/train/images/China_Dro...,China_Drone/China_Drone/train/annotations/xmls...,512,512,3.0,1,0,1,0,0,0,1,0,0,1,0,0,NaN
1,China_Drone__China_Drone_000001,China_Drone,China_Drone/China_Drone/train/images/China_Dro...,China_Drone/China_Drone/train/annotations/xmls...,512,512,3.0,2,1,1,0,0,1,1,0,0,1,0,0,NaN
2,China_Drone__China_Drone_000002,China_Drone,China_Drone/China_Drone/train/images/China_Dro...,China_Drone/China_Drone/train/annotations/xmls...,512,512,3.0,2,2,0,0,0,1,0,0,0,1,0,0,NaN
3,China_Drone__China_Drone_000003,China_Drone,China_Drone/China_Drone/train/images/China_Dro...,China_Drone/China_Drone/train/annotations/xmls...,512,512,3.0,3,1,2,0,0,1,1,0,0,1,0,0,NaN
4,China_Drone__China_Drone_000004,China_Drone,China_Drone/China_Drone/train/images/China_Dro...,China_Drone/China_Drone/train/annotations/xmls...,512,512,3.0,2,0,1,1,0,0,1,1,0,1,0,0,NaN


,sample_id,country,object_index,class,xmin,ymin,xmax,ymax,bbox_width,bbox_height,bbox_area,bbox_area_ratio
0,China_Drone__China_Drone_000000,China_Drone,0,D10,323,213,390,245,67,32,2144,0.008179
1,China_Drone__China_Drone_000001,China_Drone,0,D10,14,70,198,99,184,29,5336,0.020355
2,China_Drone__China_Drone_000001,China_Drone,1,D00,95,114,120,272,25,158,3950,0.015068
3,China_Drone__China_Drone_000002,China_Drone,0,D00,12,214,68,511,56,297,16632,0.063446
4,China_Drone__China_Drone_000002,China_Drone,1,D00,60,1,112,212,52,211,10972,0.041855


## 5. Integridad de `images.csv`

In [9]:
image_integrity = {
    "rows": len(images),
    "unique_sample_id": images["sample_id"].nunique(),
    "duplicated_sample_id": int(images["sample_id"].duplicated().sum()),
    "missing_country": int(images["country"].isna().sum()),
    "missing_image_path": int(images["image_path"].isna().sum()),
    "missing_xml_path": int(images["xml_path"].isna().sum()),
}

pd.Series(image_integrity)

rows                    38385
unique_sample_id        38385
duplicated_sample_id        0
missing_country             0
missing_image_path          0
missing_xml_path            0
dtype: int64

In [10]:
assert len(images) == 38385
assert images["sample_id"].is_unique
assert images["country"].isna().sum() == 0
assert images["image_path"].isna().sum() == 0
assert images["xml_path"].isna().sum() == 0

print("images.csv: OK")

images.csv: OK


## 6. Integridad de `annotations.csv`

In [11]:
orphan_annotations = (
    set(annotations["sample_id"])
    - set(images["sample_id"])
)

annotation_integrity = {
    "rows": len(annotations),
    "orphan_sample_ids": len(orphan_annotations),
    "missing_class": int(annotations["class"].isna().sum()),
    "invalid_bbox_width": int((annotations["bbox_width"] < 0).sum()),
    "invalid_bbox_height": int((annotations["bbox_height"] < 0).sum()),
}

pd.Series(annotation_integrity)

rows                   65712
orphan_sample_ids          0
missing_class              0
invalid_bbox_width         0
invalid_bbox_height        0
dtype: int64

In [12]:
assert len(annotations) == 65712
assert len(orphan_annotations) == 0
assert annotations["class"].isna().sum() == 0
assert (annotations["bbox_width"] < 0).sum() == 0
assert (annotations["bbox_height"] < 0).sum() == 0

print("annotations.csv: OK")

annotations.csv: OK


## 7. Clases encontradas

La tarea principal del proyecto utiliza **D40 = pothole** como etiqueta positiva. Sin embargo, el dataset contiene etiquetas adicionales. Se preservan en el manifest para no perder información y para analizar posibles confusiones.

In [13]:
class_counts = (
    annotations["class"]
    .value_counts()
    .rename_axis("class")
    .reset_index(name="objects")
)

class_counts

,class,objects
0,D00,26016
1,D10,11830
2,D20,10617
3,D40,6544
4,D44,5057
5,D50,3581
6,Repair,1046
7,D43,793
8,D01,179
9,D11,45


## 8. Resumen D40 / potholes

In [14]:
d40_summary = (
    images.groupby("country")
    .agg(
        total_images=("sample_id", "count"),
        images_with_D40=("has_D40", "sum"),
        empty_annotations=("is_empty_annotation", "sum"),
    )
    .reset_index()
)

d40_summary["images_without_D40"] = (
    d40_summary["total_images"] - d40_summary["images_with_D40"]
)

d40_summary["pothole_prevalence_pct"] = (
    100 * d40_summary["images_with_D40"] / d40_summary["total_images"]
)

d40_summary

,country,total_images,images_with_D40,empty_annotations,images_without_D40,pothole_prevalence_pct
0,China_Drone,2401,64,5,2337,2.665556
1,China_MotorBike,1977,164,0,1813,8.295397
2,Czech,2829,154,1757,2675,5.443620
3,India,7706,1530,3921,6176,19.854659
4,Japan,10506,1390,794,9116,13.230535
5,Norway,8161,256,5247,7905,3.136870
6,United_States,4805,116,0,4689,2.414152


In [15]:
total_pos = int(images["has_D40"].sum())
total_neg = int(len(images) - total_pos)
prevalence = 100 * total_pos / len(images)

print("Imágenes con D40:", total_pos)
print("Imágenes sin D40:", total_neg)
print(f"Prevalencia D40: {prevalence:.2f}%")

assert total_pos == 3674
assert total_neg == 34711

Imágenes con D40: 3674
Imágenes sin D40: 34711
Prevalencia D40: 9.57%


## 9. Validación completa opcional

La siguiente celda está desactivada por defecto porque recorre nuevamente todos los XML y puede tardar varios minutos. Actívala solo si se desea repetir la auditoría completa desde cero.

In [16]:
RUN_FULL_RAW_VALIDATION = False

if RUN_FULL_RAW_VALIDATION:
    script = PROJECT_ROOT / "src" / "data" / "validate_dataset.py"
    subprocess.run([sys.executable, str(script)], check=True)
else:
    print("Validación completa omitida. Cambia RUN_FULL_RAW_VALIDATION=True para ejecutarla.")

Validación completa omitida. Cambia RUN_FULL_RAW_VALIDATION=True para ejecutarla.


## Conclusión

Si todas las aserciones anteriores pasan, el dataset y los manifests están listos para el EDA.

**Checkpoint esperado:**
- 38,385 imágenes de entrenamiento;
- 38,385 XML;
- 9,035 imágenes de test;
- 47,420 imágenes totales;
- 38,385 filas en `images.csv`;
- 65,712 filas en `annotations.csv`;
- 3,674 imágenes con al menos un D40.